# ELEC4542 project: Can sequence models think deeper than they were taught?
Base comparison (Topic 5): RNN vs LSTM vs Transformer vs Linear Attention on ListOps-64.
Our extensions: (1) depth generalisation, (2) probing hidden states for sub-answers, (3) attention 'bracket focus', (4) our Looped Transformer that can think longer at test time.

Runtime → Change runtime type → **T4 GPU**. Upload all `.py` files (data, models, train, analyze, probe, attention, loops) to the Files panel, then run top to bottom. Total ≈ 2–3 h on a free T4.

In [ ]:
!nvidia-smi
!pip -q install datasets scikit-learn
import torch; print(torch.__version__, torch.cuda.is_available())

## 1. Data check
Label distribution, majority baseline, length/depth counts, label check. **Look at the depth counts**: you need them to pick D in step 4.

In [ ]:
!python data.py

## 2. Smoke test (catches bugs before the long runs, ~2 min)

In [ ]:
for m in ['rnn','lstm','transformer','linear','looped']:
    !python train.py --model {m} --run_name smoke_{m} --max_steps 300 --eval_every 100 --no_test --out_dir smoke

## 3. Base comparison (the brief) + our Looped Transformer
Same data, same budget (20 epochs), same optimiser, seed 0. ~15–25 min per model on a T4.

In [ ]:
for m in ['rnn','lstm','transformer','linear','looped']:
    !python train.py --model {m} --run_name base_{m}

## 4. Extension 1: think deeper than taught
Train only on depth ≤ D, test on everything. Pick D from step 1 so about 15–30% of the test set is deeper than D (default 3; change if needed).

In [ ]:
D = 3
for m in ['rnn','lstm','transformer','linear','looped']:
    !python train.py --model {m} --run_name depth{D}_{m} --train_max_depth {D}

## 5. Extension 2: mind-reading probes (~5 min)
Can a linear probe read each sub-answer (e.g. `[SM 7 6 ]` → 3) from the hidden state? Includes untrained-model controls.

In [ ]:
!python probe.py --runs runs --out figures

## 6. Extension 3: where does the Transformer look? (~2 min)

In [ ]:
!python attention.py --runs runs --out figures

## 7. Extension 4: thinking longer at test time (Looped Transformer, ~2 min)

In [ ]:
!python loops.py --run base_looped
!python loops.py --run depth3_looped

## 8. All figures and tables for the report

In [ ]:
!python analyze.py --runs runs --out figures
from IPython.display import Image, display
import glob
for f in sorted(glob.glob('figures/*.png')): print(f); display(Image(f))
for f in sorted(glob.glob('figures/*.csv'))+sorted(glob.glob('figures/*.txt')): print('=====', f); print(open(f).read())

## 9. Save everything (Colab deletes files when the runtime stops)

In [ ]:
!zip -qr results.zip runs/*/results.json runs/*/history.json runs/*/config.json runs/*/test_predictions.csv figures
from google.colab import files; files.download('results.zip')